In [1]:
import pandas as pd
from datetime import datetime
import sqlite3
import random

In [2]:
conn = sqlite3.connect("./my_database.db")

In [3]:
vendor_ids = pd.read_sql("select distinct(vendor_id) from ticket", con=conn)

In [4]:
vendor_ids

,vendor_id
0,1cfdb399-443a-4fdd-a893-38cc0ef2e515
1,9d6b94ff-9ed8-415a-bbab-98e8afb95146
2,179994ef-ab52-4434-be8c-1a46dbc79b54
3,c6fc8f7d-848c-47d7-87d6-017882e78398
4,c58489bf-7638-418b-8182-ed2b90735c9a
5,46236ae0-7e3b-4db7-a40d-a682ca3ddf5c
6,0f75bb2a-4f16-4ca1-bfc7-9df7c92b6acd
7,4d799fbf-ff2f-42ec-9b22-ea5a49123afd
8,5440af21-b03f-474c-b8b7-91c87980be87
9,efbf09e4-3c70-4a4f-8876-6bc1eb8c2763


In [5]:
test_vendor = vendor_ids.iloc[1]['vendor_id']

In [6]:
test_vendor

'9d6b94ff-9ed8-415a-bbab-98e8afb95146'

In [7]:
def active_tickets(vendor_id):
    df = pd.read_sql(f"""
    select count(*) as active_tickets
    from ticket 
    where vendor_id='{vendor_id}' and status = 'in progress'
    """, con=conn)
    count = df['active_tickets'].iloc[0]
    return {'vendor_id': vendor_id, 'active tickets': count}

In [8]:
active_tickets(test_vendor)

{'vendor_id': '9d6b94ff-9ed8-415a-bbab-98e8afb95146',
 'active tickets': np.int64(134)}

In [9]:
def vendor_on_time_delivery(vendor_id):
    df = pd.read_sql(
        f"""
        select * 
        from ticket
        where vendor_id = '{vendor_id}'
        """,
        con=conn
    )
    df['on_time'] = (
        df['completed_at'].notna() &
        (df['completed_at'] <= df['due_date'])
    )

    df = df.groupby('vendor_id', as_index=False).agg(on_time_count=("on_time", "sum"),ticket_count=("ticket_id", "count"))
    df['on_time_pct'] = df['on_time_count'] / df['ticket_count'] * 100
    print(df[['vendor_id', 'on_time_count', 'ticket_count', 'on_time_pct']])
    on_time_pct = df['on_time_pct'].iloc[0].round(2)
    return {'vendor_id': vendor_id, 'on_time_pct': on_time_pct}

In [10]:
vendor_on_time_delivery(test_vendor)

                              vendor_id  on_time_count  ticket_count  \
0  9d6b94ff-9ed8-415a-bbab-98e8afb95146            123           526   

   on_time_pct  
0     23.38403  


{'vendor_id': '9d6b94ff-9ed8-415a-bbab-98e8afb95146',
 'on_time_pct': np.float64(23.38)}

In [11]:
def flagged_tickets(vendor_id):
    df = pd.read_sql(
        f"""
            select count(*) as flagged_tickets from ticket
            where vendor_id='{vendor_id}' and anomaly_flag=True
        """, con=conn
    )
    flagged_ticket_count = df['flagged_tickets'].iloc[0]
    return {'vendor_id': vendor_id, 'flagged_ticket_count': flagged_ticket_count}

In [12]:
flagged_tickets(test_vendor)

{'vendor_id': '9d6b94ff-9ed8-415a-bbab-98e8afb95146',
 'flagged_ticket_count': np.int64(265)}

## Completed Today (work in progress due to logic of sample data)

In [13]:
df = pd.read_sql(
        f"""
        select *
        from ticket
        where vendor_id='{test_vendor}' and status='completed'
        """, parse_dates=['assigned_at', 'completed_at', 'created_at', 'updated_at', 'start_time', 'due_date'], con=conn
    )

In [14]:
max(df['completed_at'])

Timestamp('2026-04-08 06:01:44')

In [15]:
def completed_tickets_today(vendor_id):
    df = pd.read_sql(
        f"""
        select *
        from ticket
        where vendor_id='{vendor_id}'
        """, con=conn
    )
    return df

In [16]:
# completed_tickets_today(test_vendor)

## Average Duration

In [17]:
def average_duration(vendor_id):
    df = pd.read_sql(
        f"""
        select * from ticket
        where vendor_id = '{vendor_id}' and status='completed'
        """,
        con=conn,
        parse_dates=['completed_at', 'start_time']
    )
    df['duration'] = df['completed_at'] - df['start_time']
    avg_duration = df['duration'].mean()
    return {'vendor_id': vendor_id, 'avg_duration': avg_duration}

In [18]:
average_duration(test_vendor)

{'vendor_id': '9d6b94ff-9ed8-415a-bbab-98e8afb95146',
 'avg_duration': Timedelta('2 days 12:14:48.408536')}